# Information Theory in LLMs

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 05.06 |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/05_Information_Theory/07_information_theory_in_llms.ipynb)

---

## 🎯 Learning Objective

Connect perplexity and bits-per-character to entropy, and use them to evaluate language models and interpret scaling laws.

---

## 📐 Theory

This capstone notebook connects every idea in Module 05 to the two questions that actually
drive LLM development in practice: *how do you measure whether a language model is good?* and
*how do you decide how big to make it, and how much data to train it on?*

### Perplexity: cross-entropy, exponentiated

Language models are evaluated with **perplexity**, defined from the model's average
cross-entropy loss (`05.03`) on held-out text:

$$\text{PPL} = 2^{H(P,Q)} \quad\text{(if } H \text{ measured in bits)} \qquad\text{or}\qquad \text{PPL} = e^{H(P,Q)} \quad\text{(if } H \text{ measured in nats)}$$

Both formulas describe the *same* quantity — perplexity doesn't care which log base you used to
compute cross-entropy, because exponentiating with the matching base always undoes it exactly
(we verify this numerically below). Why exponentiate at all, rather than just reporting
cross-entropy directly? Perplexity has a clean interpretation as an **effective vocabulary
size**: a model with perplexity 20 is, on average, as uncertain about the next token as if it
were choosing uniformly at random among 20 equally likely options. A perfect model that always
assigns probability 1 to the true next token has cross-entropy 0 and perplexity $2^0=1$ — no
uncertainty at all. This is a much more intuitive number than a raw bits value, which is why
papers report "perplexity 12.3" rather than "cross-entropy 3.62 nats," even though the latter is
what was actually computed.

### Bits-per-character and tokenizer-independent comparison

Perplexity has an awkward property: it's defined *per token*, and different models use
different tokenizers (`05.06`) with different vocabularies and different average
characters-per-token. A model with a coarser tokenizer (fewer, longer tokens) will report a
*lower* perplexity for reasons that have nothing to do with how good it is at modeling language.
**Bits-per-character (bpc)** fixes this by normalizing cross-entropy to a tokenizer-independent
unit: divide the total bits spent by the total number of *characters* (not tokens) in the
evaluated text. This is exactly why Module 05.06's framing of entropy as a per-symbol quantity
matters — bpc is entropy estimated at the character level, letting you fairly compare models
that tokenize text completely differently.

### The Chinchilla scaling law

Given a fixed training compute budget $C$ (roughly $C \approx 6ND$ floating-point operations,
where $N$ is parameter count and $D$ is training tokens), how should you split compute between
making the model bigger versus training it on more data? Hoffmann et al. (2022, DeepMind)
answered this by fitting a parametric loss model across hundreds of training runs:

$$L(N,D) = E + \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}}$$

Every symbol here is directly interpretable: $E$ is the **irreducible entropy of natural
text** — literally the $H(P)$ floor from `05.03`'s identity $H(P,Q)=H(P)+D_{KL}(P\|Q)$, the loss
even an infinitely large model trained on infinite data could never beat. The term $A/N^\alpha$
is the penalty for having *finite model capacity* (a real transformer can't perfectly represent
the true text distribution); $B/D^\beta$ is the penalty for *finite training data* (even a
perfect-capacity model trained on too little data hasn't converged). Hoffmann et al. fit
$E{=}1.69$, $A{=}406.4$, $B{=}410.7$, $\alpha{=}0.34$, $\beta{=}0.28$, with loss measured in
nats (the paper's training loss is the standard natural-log cross-entropy, per `05.03`'s
base-conversion note).

Minimizing $L(N,D)$ subject to the compute constraint $C=6ND$ has a closed-form solution: both
$N_{\text{opt}}(C)$ and $D_{\text{opt}}(C)$ scale as *power laws* in $C$, with exponents
$a=\beta/(\alpha+\beta)$ and $b=\alpha/(\alpha+\beta)$ respectively. Plugging in the fitted
$\alpha,\beta$ gives $a\approx0.45$ and $b\approx0.55$ — close enough to $0.5$ each that
Hoffmann et al. describe the headline finding as **compute-optimal training scales model size
and data size at roughly equal rates**, in sharp contrast to the field's prior practice
(exemplified by GPT-3-scale models), where most large models were substantially *undertrained*
relative to their size. Because $b$ is slightly larger than $a$, the *ratio* $D_{\text{opt}}/N_{\text{opt}}$
(tokens per parameter) isn't perfectly constant — it grows slowly as $C$ increases, something
we'll see directly in the exercises. We derive and solve this optimization numerically below,
reproducing the headline Chinchilla-vs-Gopher comparison with a worked example.

---

In [ ]:
# Setup
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Perplexity is an exponential function of cross-entropy, so it drops much faster than the loss
curve near the start of training and flattens out later -- the two views of "how good is the
model" look qualitatively different even though they carry identical information.

In [ ]:
# Simulate a typical training loss curve (cross-entropy in nats, decreasing over training steps)
# and show how perplexity = e^loss transforms it -- same information, very different shape
training_steps = np.linspace(0, 1, 200)
simulated_loss_nats = 0.3 + 3.7 * np.exp(-4 * training_steps)   # a plausible smooth training curve
perplexity = np.exp(simulated_loss_nats)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(training_steps, simulated_loss_nats, color="#4C72B0", lw=2)
axes[0].set_xlabel("training progress"); axes[0].set_ylabel("cross-entropy loss [nats]")
axes[0].set_title("Loss decreases roughly linearly\n(on a log-ish schedule)")

axes[1].plot(training_steps, perplexity, color="#DD8452", lw=2)
axes[1].set_xlabel("training progress"); axes[1].set_ylabel("perplexity = e^(loss)")
axes[1].set_title("Perplexity drops MUCH faster early on\n(exponential of the loss curve)")
plt.tight_layout()
plt.show()

print(f"{'progress':>10} | {'loss (nats)':>12} | {'perplexity':>11}")
for frac in [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]:
    idx = int(frac * (len(training_steps) - 1))
    print(f"{training_steps[idx]:10.2f} | {simulated_loss_nats[idx]:12.3f} | {perplexity[idx]:11.2f}")
print("\nEarly training reduces perplexity dramatically for each unit of loss improvement;")
print("later training reduces perplexity only slightly for the same size loss improvement --")
print("a direct consequence of exponentiating a mostly-linear curve.")

## 🐍 Implementation from Scratch

We compute cross-entropy and perplexity directly on a toy model's predictions, confirm base-2
and base-$e$ give identical perplexity, then implement the Chinchilla scaling law's
compute-optimal frontier from its closed-form solution and verify it against a brute-force grid
search over model/data splits.

In [ ]:
# --- Perplexity from cross-entropy, confirmed identical regardless of log base used ---
rng = np.random.default_rng(0)
vocab_size, seq_len = 20, 50
true_tokens = rng.integers(0, vocab_size, size=seq_len)
logits = rng.normal(scale=1.0, size=(seq_len, vocab_size))
logits[np.arange(seq_len), true_tokens] += 3.0   # a partially-skilled toy model

probs = F.softmax(torch.tensor(logits), dim=1).numpy()
true_token_probs = probs[np.arange(seq_len), true_tokens]

H_bits = -np.mean(np.log2(true_token_probs))
H_nats = -np.mean(np.log(true_token_probs))
perplexity_from_bits = 2 ** H_bits
perplexity_from_nats = np.e ** H_nats

print(f"Cross-entropy: {H_bits:.4f} bits/token  =  {H_nats:.4f} nats/token  (ratio = ln(2) = {np.log(2):.4f})")
print(f"Perplexity via 2^(bits):  {perplexity_from_bits:.4f}")
print(f"Perplexity via e^(nats):  {perplexity_from_nats:.4f}")
print(f"Match: {np.isclose(perplexity_from_bits, perplexity_from_nats)}  "
      f"(both log bases describe the identical model, so perplexity must agree)")
print(f"For reference, a uniformly random model over {vocab_size} tokens has perplexity {vocab_size}"
      f" -- this model (perplexity {perplexity_from_nats:.1f}) is meaningfully better than chance.")

# --- Chinchilla scaling law: closed-form compute-optimal frontier vs brute-force grid search ---
E, A, B, alpha, beta = 1.69, 406.4, 410.7, 0.34, 0.28   # Hoffmann et al. (2022) fitted constants

def chinchilla_loss(N, D):
    return E + A / N ** alpha + B / D ** beta

def compute_optimal_closed_form(C):
    # Solving min_{N,D: 6ND=C} L(N,D) analytically gives power laws in C (Hoffmann et al. eq. 4)
    a_exp, b_exp = beta / (alpha + beta), alpha / (alpha + beta)
    G = (alpha * A / (beta * B)) ** (1 / (alpha + beta))
    N_opt = G * (C / 6) ** a_exp
    D_opt = (C / 6) ** b_exp / G
    return N_opt, D_opt

def compute_optimal_grid_search(C, n_grid=4000):
    # Brute-force check: sweep N over a wide range, set D=C/(6N) to respect the compute
    # constraint exactly, and find whichever N minimizes loss -- an independent verification
    N_grid = np.logspace(8, 13, n_grid)
    D_grid = C / (6 * N_grid)
    losses = chinchilla_loss(N_grid, D_grid)
    best = np.argmin(losses)
    return N_grid[best], D_grid[best]

C_gopher = 5.76e23   # Gopher's actual training compute, as reported in the Chinchilla paper
N_cf, D_cf = compute_optimal_closed_form(C_gopher)
N_gs, D_gs = compute_optimal_grid_search(C_gopher)

print(f"\nAt Gopher's compute budget C={C_gopher:.3e} FLOPs:")
print(f"  closed-form optimum:   N={N_cf:.3e}  D={D_cf:.3e}  (tokens/param = {D_cf/N_cf:.1f})")
print(f"  grid-search optimum:   N={N_gs:.3e}  D={D_gs:.3e}  (tokens/param = {D_gs/N_gs:.1f})")
print(f"  agreement: {np.isclose(N_cf, N_gs, rtol=1e-2)} (two independent methods, same answer)")

## 🤖 Why This Matters for AI

Every LLM leaderboard, model card, and training log reports perplexity or a close relative
(bits-per-byte, loss) as the primary signal of model quality -- it's the single number that
most directly answers "how surprised is this model by real text?" The Chinchilla scaling law
reshaped how every major lab plans training runs after 2022: instead of asking "how large can I
afford to make my model," teams now ask "what's the compute-optimal *split* between size and
data," a direct consequence of treating $N$ and $D$ as jointly optimized rather than $N$ as the
only lever worth pulling. Below we reproduce the paper's central comparison -- Gopher (280B
params, undertrained) versus Chinchilla (70B params, compute-matched but far better-trained) --
with a worked numeric example showing why the smaller, better-trained model wins.

The tools that compute and report these metrics: EleutherAI's `lm-evaluation-harness` is the standard benchmark suite, and the **Hugging Face Open LLM Leaderboard** ranks models by exactly the perplexity and downstream task scores derived from the information-theoretic quantities above.

---


In [ ]:
# Reproduce the Chinchilla paper's headline comparison: same training compute, very different
# (N, D) split. Gopher spent its budget almost entirely on PARAMETERS; Chinchilla, following the
# compute-optimal frontier derived above, spent proportionally more on DATA.
N_gopher, D_gopher = 280e9, 300e9      # Gopher: 280B parameters, 300B tokens (as actually trained)
N_chinchilla, D_chinchilla = 70e9, 1.4e12   # Chinchilla: 70B parameters, 1.4T tokens

C_gopher_actual = 6 * N_gopher * D_gopher
C_chinchilla_actual = 6 * N_chinchilla * D_chinchilla

print(f"{'Model':<12}{'Params':>10}{'Tokens':>12}{'Compute (FLOPs)':>18}{'Loss (nats)':>14}")
for name, N, D, C in [("Gopher", N_gopher, D_gopher, C_gopher_actual),
                       ("Chinchilla", N_chinchilla, D_chinchilla, C_chinchilla_actual)]:
    print(f"{name:<12}{N:>10.2e}{D:>12.2e}{C:>18.3e}{chinchilla_loss(N, D):>14.4f}")

print(f"\nUsing the simple C=6ND approximation, Gopher and Chinchilla used roughly comparable")
print(f"compute ({C_gopher_actual:.2e} vs {C_chinchilla_actual:.2e} FLOPs, a "
      f"{C_chinchilla_actual/C_gopher_actual:.2f}x ratio -- the paper's own more detailed FLOP")
print(f"accounting, which we don't reproduce here, puts their ACTUAL budgets even closer). Even")
print(f"so, Chinchilla's predicted loss is LOWER, from redistributing compute toward more training")
print(f"tokens per parameter rather than more parameters -- the scaling law's central, testable")
print(f"claim. (This is the FITTED law's predicted loss, not the paper's measured evaluation loss")
print(f"-- the real paper confirms the same direction empirically across many benchmarks.)")

# --- Worked example: given a NEW compute budget, find the compute-optimal model+data split ---
C_new = 1e25   # roughly 17x Gopher's budget -- a plausible "next generation" compute budget
N_new_opt, D_new_opt = compute_optimal_closed_form(C_new)
print(f"\nWorked example -- planning a new run with C={C_new:.1e} FLOPs (~{C_new/C_gopher_actual:.1f}x Gopher):")
print(f"  compute-optimal model size:   {N_new_opt:.3e} parameters ({N_new_opt/1e9:.1f}B)")
print(f"  compute-optimal training set: {D_new_opt:.3e} tokens ({D_new_opt/1e12:.2f}T)")
print(f"  predicted loss at this optimum: {chinchilla_loss(N_new_opt, D_new_opt):.4f} nats")
print(f"  tokens per parameter: {D_new_opt/N_new_opt:.1f}")
print(f"  (this ratio GROWS with compute under approach-3's fitted exponents -- it is not the")
print(f"  single constant '20' popularly quoted, which describes Chinchilla's own compute scale;")
print(f"  see Exercise 2 below for how the ratio shifts across budgets)")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. A model achieves a cross-entropy loss of 2.0 nats/token on a test set. Compute its perplexity
   by hand, then verify with `np.exp`. Separately, what cross-entropy in BITS/token would give
   the same perplexity? Verify with `2**`. As a sanity check, confirm both cross-entropy values
   are non-negative and that perplexity is at least 1 (a model can never be *less* uncertain than
   total certainty).

<details>
<summary>💡 Solution</summary>

$\text{PPL} = e^{2.0} \approx 7.389$. `np.exp(2.0)` confirms this. For the equivalent bits value,
solve $2^{H_{\text{bits}}} = 7.389 \implies H_{\text{bits}} = \log_2(7.389) \approx 2.885$
bits/token — and indeed $2.0/\ln 2 \approx 2.885$ matches directly, since converting nats to
bits is just dividing by $\ln 2$. `2**2.885` returns `7.389`, matching the nats-based perplexity
exactly, confirming both log bases describe the identical model. Both cross-entropy values
($2.0$ nats, $2.885$ bits) are positive, and perplexity $7.389 \ge 1$, as required — perplexity
$1$ would mean a perfectly confident, always-correct model.

</details>

### 💭 UNDERSTAND
2. Model X reports perplexity 15 using a tokenizer that averages 4 characters/token. Model Y
   reports perplexity 25 using a *coarser* tokenizer averaging 6 characters/token. Based purely
   on their reported perplexities, which model looks better? Now convert both to bits-per-
   character (recall $\text{bpc} = \log_2(\text{PPL})/\text{chars per token}$) and compare
   again. Does the ranking change?

<details>
<summary>💡 Solution</summary>

By perplexity alone, Model X looks better ($15 < 25$). But converting to bits-per-character:
Model X gives $\log_2(15)/4 \approx 0.977$ bpc, while Model Y gives $\log_2(25)/6 \approx
0.774$ bpc — Model Y is actually *better* once you normalize for tokenizer coarseness. The
ranking flips entirely. This is exactly the Theory section's warning made concrete: Model Y's
higher reported perplexity is an artifact of spending more characters per token (so each
"token-level mistake" is amortized over more characters), not evidence of a worse underlying
model. Comparing raw perplexity across models with different tokenizers is comparing different
units dressed up as the same number — bits-per-character is the fair, tokenizer-independent
comparison.

</details>

### ✏️ DERIVE
3. Starting from $L(N,D) = E + A/N^\alpha + B/D^\beta$ and the compute constraint $C=6ND$,
   derive that the compute-optimal $N_{\text{opt}}(C)$ scales as $C^{\beta/(\alpha+\beta)}$.
   (Hint: substitute $D=C/(6N)$ to make $L$ a function of $N$ alone, then set
   $\frac{dL}{dN}=0$.)

<details>
<summary>💡 Solution outline</summary>

Substituting $D=C/(6N)$: $L(N) = E + AN^{-\alpha} + B\big(\tfrac{6N}{C}\big)^{\beta} = E +
AN^{-\alpha} + B\,6^\beta C^{-\beta} N^{\beta}$. Differentiating with respect to $N$:
$$\frac{dL}{dN} = -\alpha A N^{-\alpha-1} + \beta B\,6^\beta C^{-\beta} N^{\beta - 1}$$
Setting this to zero and solving for $N$: $\alpha A N^{-\alpha-1} = \beta B\,6^\beta
C^{-\beta}N^{\beta-1}$, so $N^{-\alpha-1-(\beta-1)} = N^{-\alpha-\beta} =
\frac{\beta B\,6^\beta}{\alpha A}C^{-\beta}$. Taking both sides to the power
$-1/(\alpha+\beta)$:
$$N_{\text{opt}} = \left(\frac{\alpha A}{\beta B\,6^\beta}\right)^{1/(\alpha+\beta)} C^{\beta/(\alpha+\beta)}$$
which is exactly a power law in $C$ with exponent $\beta/(\alpha+\beta)$, as claimed — matching
`compute_optimal_closed_form`'s `a_exp = beta / (alpha + beta)`. Plugging in the fitted
$\alpha=0.34,\beta=0.28$ gives exponent $\approx 0.452$. A symbolic check (e.g. with `sympy`,
differentiating $L(N)$ and confirming the derivative vanishes at the closed-form $N_{\text{opt}}$
for a specific numeric $C$) confirms this algebra numerically to machine precision.

</details>

### 🐍 IMPLEMENT
4. Simulate a small "scaling law experiment": sample 200 random $(N,D)$ pairs (e.g.
   log-uniformly over realistic ranges), compute each pair's loss with `chinchilla_loss`, add
   small Gaussian noise (representing real experimental variance), and use
   `scipy.optimize.curve_fit` to *recover* $E,A,B,\alpha,\beta$ from the noisy
   $(N,D,\text{loss})$ observations alone — treating the true Hoffmann et al. constants as
   unknown ground truth to be rediscovered, the way the original paper had to fit them from real
   training runs.

<details>
<summary>✅ How to know you're right</summary>

With 200 samples and modest noise (e.g. standard deviation $0.01$ nats), `curve_fit` should
recover all five parameters close to their true values — a representative run recovers
$E\approx 1.69$ (matching almost exactly, since $E$ is a simple additive offset), and
$A,B,\alpha,\beta$ within roughly 5-10% of their true values ($A{=}406.4, B{=}410.7,
\alpha{=}0.34,\beta{=}0.28$ in the notebook's convention). Because $N$ and $D$ span several
orders of magnitude in your sample (log-uniform sampling is essential here — sampling
$N,D$ *linearly* over a wide range would concentrate almost all samples at the largest values and
starve the fit of information about the small-$N$/small-$D$ regime where $A/N^\alpha$ and
$B/D^\beta$ actually vary), the fit should be well-conditioned. If `curve_fit` fails to converge
or returns wildly wrong values, the most likely cause is an uninformative initial guess (`p0`) or
too narrow a sampled range for $N,D$, not a fundamental problem with the method.

</details>

### 🤖 APPLY
5. Using `chinchilla_loss` and `compute_optimal_closed_form`, compute the compute-optimal
   $(N,D)$ split for $C=10^{22}$ and $C=10^{26}$ FLOPs. Report the tokens-per-parameter ratio at
   each budget, and describe in one sentence how that ratio changes as compute grows (hint: it
   is NOT constant, even though $N_{\text{opt}}$ and $D_{\text{opt}}$ both scale as
   $\approx\sqrt{C}$).

<details>
<summary>✅ What you should observe</summary>

At $C=10^{22}$: $N_{\text{opt}}\approx 5.16\text{B}$ parameters, $D_{\text{opt}}\approx
0.32\text{T}$ tokens, a ratio of about $63$ tokens/parameter. At $C=10^{26}$ (four orders of
magnitude more compute): $N_{\text{opt}}\approx 330\text{B}$ parameters,
$D_{\text{opt}}\approx 50\text{T}$ tokens, a ratio of about $153$ tokens/parameter — roughly
$2.4\times$ higher. The ratio grows as a power law in $C$ with exponent
$\beta/(\alpha+\beta) - \alpha/(\alpha+\beta) \approx 0.097$ (small but clearly nonzero,
since $\beta<\alpha$ makes $D_{\text{opt}}$ scale very slightly faster than $N_{\text{opt}}$)
— compute-optimal training doesn't just scale $N$ and $D$ at the same rate forever, it gradually
tilts toward relatively more data as budgets grow, which is why the popular "20 tokens per
parameter" heuristic is only a snapshot near Chinchilla's own compute scale, not a universal
constant.

</details>

### 🚀 CHALLENGE
6. Suppose you're compute-constrained to $C=3\times10^{23}$ FLOPs (about half of Gopher's
   budget) but need a model with AT LEAST 100B parameters for some deployment reason unrelated
   to training loss (e.g. a fixed architecture requirement). Using `chinchilla_loss`, compute
   the loss you'd get from the compute-optimal $(N,D)$ split versus the loss from spending the
   *same* compute on $N=100\text{B}$ with whatever $D$ that leaves you (via $D=C/6N$). Quantify
   how much loss you sacrifice by being forced away from the compute-optimal frontier, and
   discuss why real deployment constraints (memory, latency) sometimes make this trade
   unavoidable in practice.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer computes the compute-optimal split at $C=3\times10^{23}$ as roughly
$N_{\text{opt}}\approx 24\text{B}$ parameters and $D_{\text{opt}}\approx 2.09\text{T}$
tokens, achieving loss $\approx 1.956$ nats. Forced to $N=100\text{B}$ (about $4\times$ larger
than optimal) at the same compute budget, the leftover training data shrinks to
$D\approx 0.5\text{T}$ tokens, giving loss $\approx 1.982$ nats — a sacrifice of about $0.026$
nats, roughly $1.3\%$ relative. A strong answer notes this is a genuinely *small* penalty given
how far $N=100\text{B}$ sits from the optimum, because the loss surface is fairly flat near its
minimum along the compute-constrained frontier (part of why Hoffmann et al.'s own paper notes a
wide range of $(N,D)$ splits near the optimum perform similarly) — and connects this to real
deployment reasoning: a $\sim$1% loss sacrifice is often a completely acceptable price for a
model size chosen to fit a specific GPU's memory, meet a latency SLA, or match an existing
serving architecture, since those constraints don't bend to what a scaling law says is
"optimal." A strong answer also flags that this conclusion is specific to how *close* the forced
$N$ is to $N_{\text{opt}}$ — forcing $N$ far more extreme (e.g. $N=1\text{B}$ or $N=10\text{T}$
at this same compute budget) would show a much steeper loss penalty, since the flat region near
the optimum doesn't extend indefinitely.

</details>

---

## 📚 Further Reading
- Hoffmann et al., ["Training Compute-Optimal Large Language Models"](https://arxiv.org/abs/2203.15556) (the Chinchilla paper)
- Kaplan et al., ["Scaling Laws for Neural Language Models"](https://arxiv.org/abs/2001.08361) (the earlier scaling law this one revised)
- Jurafsky & Martin, *Speech and Language Processing*, Ch. 3 (N-gram Language Models and perplexity)

---

*Next notebook: [Floating-Point Arithmetic](../06_Numerical_Methods/01_floating_point_arithmetic.ipynb)*